In [1]:
!pip install -U transformers accelerate sentencepiece protobuf nltk
!pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 83.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 67.5 MB/s eta 0:00:00:00:01
  Attempting uninstall: sentencepiece
    Found existing installation: sentencepiece 0.2.1
    Uninstalling sentencepiece-0.2.1:
      Successfully uninstalled sentencepiece-0.2.1
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: protobuf
    Found existing installation: pr

In [2]:
import torch
from pathlib import Path
from tqdm.auto import tqdm
from IPython.display import FileLink, display
from transformers import AutoProcessor

# Keep "llava" to reproduce this notebook's existing VLM.
# Other options: "qwen3vl", "clip_vit_b32", "clip_vit_b16"
VLM_CHOICE = "llava"

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("Torch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


VLM_REGISTRY = {
    "qwen3vl": {
        "type": "vlm_lm",
        "model_id": "Qwen/Qwen3-VL-2B-Instruct",
        "output_prefix": "qwen3vl",
    },
    "llava": {
        "type": "vlm_lm",
        "model_id": "llava-hf/llava-1.5-7b-hf",
        "output_prefix": "llava15",
    },
    "clip_vit_b32": {
        "type": "hf_clip",
        "model_id": "openai/clip-vit-base-patch32",
        "output_prefix": "clip_vit_b32",
        "text_len": 77,
    },
    "clip_vit_b16": {
        "type": "hf_clip",
        "model_id": "openai/clip-vit-base-patch16",
        "output_prefix": "clip_vit_b16",
        "text_len": 77,
    },
}


def load_vlm(vlm_choice):
    cfg = VLM_REGISTRY[vlm_choice]
    model_id = cfg["model_id"]

    if cfg["type"] == "vlm_lm":
        if vlm_choice == "qwen3vl":
            from transformers import (
                Qwen3VLForConditionalGeneration,
            )
            model_class = Qwen3VLForConditionalGeneration

        elif vlm_choice == "llava":
            from transformers import (
                LlavaForConditionalGeneration,
            )
            model_class = LlavaForConditionalGeneration

        else:
            raise ValueError(vlm_choice)

        model = model_class.from_pretrained(
            model_id,
            torch_dtype="auto",
            device_map="auto" if torch.cuda.is_available() else None,
        )

    else:
        from transformers import CLIPModel

        dtype = (
            torch.float16
            if torch.cuda.is_available()
            else torch.float32
        )

        model = CLIPModel.from_pretrained(
            model_id,
            torch_dtype=dtype,
        ).to(device)

    processor = AutoProcessor.from_pretrained(model_id)
    tokenizer = processor.tokenizer
    model.eval()

    return model, processor, tokenizer, cfg


model, processor, tokenizer, vlm_cfg = load_vlm(VLM_CHOICE)

print("Loaded:", vlm_cfg["model_id"])
print("Model class:", type(model).__name__)

Torch: 2.10.0+cpu
Device: cpu


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Loaded: llava-hf/llava-1.5-7b-hf
Model class: LlavaForConditionalGeneration


In [3]:
import torch.nn.functional as F


def get_text_embedding_layer(model):
    candidates = [
        lambda m: m.model.language_model.embed_tokens,
        lambda m: m.language_model.model.embed_tokens,
        lambda m: m.language_model.embed_tokens,
        lambda m: m.get_input_embeddings(),
    ]

    for getter in candidates:
        try:
            layer = getter(model)
            if layer is not None:
                return layer
        except AttributeError:
            pass

    raise AttributeError(
        "Could not locate the VLM text embedding layer."
    )


assert vlm_cfg["type"] == "vlm_lm", (
    "This SANA projection extraction expects llava or qwen3vl."
)

embed_tokens = get_text_embedding_layer(model)
embed_device = next(embed_tokens.parameters()).device

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = (
        tokenizer.eos_token or tokenizer.unk_token
    )

tokenizer.padding_side = "right"

assert tokenizer.is_fast, (
    "A fast tokenizer is required for class-span extraction."
)

representation_name = (
    "input_embedding_class_token_span_"
    "normalized_template_mean"
)

print("Embedding device:", embed_device)
print("Embedding dimension:", embed_tokens.weight.shape[1])


@torch.inference_mode()
def encode_class_spans(
    texts,
    spans,
    batch_size=512,
    max_length=128,
):
    results = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Encoding templated classes",
    ):
        end = min(start + batch_size, len(texts))

        batch_texts = texts[start:end]
        batch_spans = spans[start:end]

        encoded = tokenizer(
            batch_texts,
            add_special_tokens=False,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        offsets = encoded.pop("offset_mapping")
        attention_mask = encoded["attention_mask"].bool()

        input_ids = encoded["input_ids"].to(embed_device)
        token_embeddings = embed_tokens(input_ids).float()

        span_starts = torch.tensor(
            [span[0] for span in batch_spans],
            dtype=torch.long,
        )[:, None]

        span_ends = torch.tensor(
            [span[1] for span in batch_spans],
            dtype=torch.long,
        )[:, None]

        token_starts = offsets[:, :, 0]
        token_ends = offsets[:, :, 1]

        span_mask = (
            attention_mask
            & (token_ends > span_starts)
            & (token_starts < span_ends)
            & (token_ends > token_starts)
        )

        missing = (~span_mask.any(dim=1)).nonzero(
            as_tuple=False
        ).flatten()

        if len(missing):
            row = int(missing[0])
            raise RuntimeError(
                "No token overlaps the class span for "
                f"{batch_texts[row]!r}"
            )

        weights = span_mask.to(
            device=token_embeddings.device,
            dtype=token_embeddings.dtype,
        ).unsqueeze(-1)

        features = (
            token_embeddings * weights
        ).sum(dim=1)

        features = features / (
            weights.sum(dim=1).clamp_min(1)
        )

        results.append(features.cpu())

    return torch.cat(results, dim=0)


def build_class_prototypes(
    class_names,
    templates,
    batch_size=512,
):
    texts = []
    spans = []

    # Class-major, then template-major.
    for class_name in class_names:
        for template in templates:
            prefix, _ = template.split("{}")
            text = template.format(class_name)

            class_start = len(prefix)
            class_end = class_start + len(class_name)

            texts.append(text)
            spans.append((class_start, class_end))

    prompt_features = encode_class_spans(
        texts,
        spans,
        batch_size=batch_size,
    )

    hidden_dim = prompt_features.shape[-1]

    prompt_features = prompt_features.reshape(
        len(class_names),
        len(templates),
        hidden_dim,
    )

    # Normalize each template, average templates per class,
    # and normalize the final class prototype.
    prompt_features = F.normalize(
        prompt_features,
        dim=-1,
    )

    class_embeddings = prompt_features.mean(dim=1)

    return F.normalize(
        class_embeddings,
        dim=-1,
    ).contiguous()

Embedding device: cpu
Embedding dimension: 4096


In [6]:
import hashlib
import json


def load_torch_file(path):
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        return torch.load(path, map_location="cpu")


search_roots = [
    Path("/kaggle/input/datasets/aveenhussein/classes/"),
    Path("/kaggle/working/datasets/aveenhussein/classes/"),
    Path.cwd(),
]

matches = []

for root in search_roots:
    if root.exists():
        matches.extend(
            root.rglob("sana_class_embeddings.pt")
        )

matches = list(
    dict.fromkeys(path.resolve() for path in matches)
)

assert len(matches) == 1, (
    "Expected exactly one sana_class_embeddings.pt, "
    f"but found: {matches}"
)

SANA_PATH = matches[0]
sana = load_torch_file(SANA_PATH)

required_keys = {
    "class_names",
    "templates",
    "embeddings",
    "class_spec",
    "model_id",
    "representation",
}

missing = required_keys - set(sana)
assert not missing, f"Missing SANA keys: {sorted(missing)}"

class_names = list(sana["class_names"])
templates = list(sana["templates"])
sana_embeddings = sana["embeddings"].float()
class_spec = sana["class_spec"]

assert class_names == class_spec["class_names"]
assert templates == class_spec["templates"]
assert len(class_names) == class_spec["class_count"]
assert len(templates) == class_spec["template_count"]
assert sana_embeddings.ndim == 2
assert sana_embeddings.shape[0] == len(class_names)
assert sana_embeddings.shape[1] == 2304
assert torch.isfinite(sana_embeddings).all()

# Stable identifiers for checking row alignment later.
class_ids = [
    hashlib.sha256(name.encode("utf-8")).hexdigest()
    for name in class_names
]

class_spec_bytes = json.dumps(
    class_spec,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

class_spec_sha256 = hashlib.sha256(
    class_spec_bytes
).hexdigest()

print("SANA file:", SANA_PATH)
print("Classes:", len(class_names))
print("Templates per class:", len(templates))
print("SANA embeddings:", tuple(sana_embeddings.shape))
print("SANA model:", sana["model_id"])
print("Representation:", sana["representation"])
print("Class specification SHA-256:", class_spec_sha256)
print("First classes:", class_names[:10])

SANA file: /kaggle/input/datasets/aveenhussein/classes/sana_class_embeddings.pt
Classes: 3000
Templates per class: 5
SANA embeddings: (3000, 2304)
SANA model: Efficient-Large-Model/gemma-2-2b-it
Representation: last_hidden_state_class_token_span_normalized_template_mean
Class specification SHA-256: 9c8341e536823b3a70ec56f8a34217afca315447dc22e185bf83a05701bc5999
First classes: ['entity', 'physical entity', 'abstraction', 'thing', 'object', 'whole', 'congener', 'living thing', 'organism', 'benthos']


In [7]:
import time

assert "build_class_prototypes" in globals(), (
    "Run the replacement Cell 2 first."
)

BATCH_SIZE = 512

start_time = time.perf_counter()

vlm_embeddings = build_class_prototypes(
    class_names,
    templates,
    batch_size=BATCH_SIZE,
)

elapsed_seconds = time.perf_counter() - start_time

assert vlm_embeddings.ndim == 2
assert vlm_embeddings.shape[0] == len(class_names)
assert sana_embeddings.shape[0] == len(class_names)
assert torch.isfinite(vlm_embeddings).all()

OUTPUT_PATH = Path(
    "/kaggle/working/"
    f"{vlm_cfg['output_prefix']}_"
    "sana_class_projection_inputs.pt"
)

output_bundle = {
    "format_version": 1,
    "kind": "paired_class_projection_inputs",

    # Inputs for calculating W.
    "X": vlm_embeddings.float().cpu(),
    "Y": sana_embeddings.float().cpu(),

    "class_names": class_names,
    "templates": templates,
    "ids": class_ids,
    "class_spec": class_spec,
    "class_spec_sha256": class_spec_sha256,

    "source_model": vlm_cfg["model_id"],
    "source_representation": representation_name,
    "target_model": sana["model_id"],
    "target_representation": sana["representation"],

    "class_count": len(class_names),
    "template_count": len(templates),
    "source_hidden_dim": int(vlm_embeddings.shape[1]),
    "target_hidden_dim": int(sana_embeddings.shape[1]),
    "extraction_seconds": elapsed_seconds,
}

torch.save(output_bundle, OUTPUT_PATH)

# Verify the saved file.
check = load_torch_file(OUTPUT_PATH)

assert check["class_names"] == class_names
assert check["templates"] == templates
assert check["ids"] == class_ids
assert check["X"].shape == vlm_embeddings.shape
assert check["Y"].shape == sana_embeddings.shape
assert torch.isfinite(check["X"]).all()
assert torch.isfinite(check["Y"]).all()

print("Saved and verified:", OUTPUT_PATH)
print("X / VLM:", tuple(check["X"].shape))
print("Y / SANA:", tuple(check["Y"].shape))
print("Classes:", len(check["class_names"]))
print("Templates:", len(check["templates"]))
print("Time:", round(elapsed_seconds, 2), "seconds")

display(FileLink(str(OUTPUT_PATH)))

Encoding templated classes:   0%|          | 0/30 [00:00<?, ?it/s]

Saved and verified: /kaggle/working/llava15_sana_class_projection_inputs.pt
X / VLM: (3000, 4096)
Y / SANA: (3000, 2304)
Classes: 3000
Templates: 5
Time: 10.48 seconds


/kaggle/working/llava15_sana_class_projection_inputs.pt

new

In [9]:
bundle = torch.load(
    "/kaggle/working/llava15_sana_class_projection_inputs.pt",
    map_location="cpu",
    weights_only=False,
)

X = bundle["X"].float()  # LLaVA, approximately [3000, 4096]
Y = bundle["Y"].float()  # SANA, [3000, 2304]

assert X.shape[0] == Y.shape[0] == 3000

In [2]:
import hashlib
import json
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm


def find_one(filename):
    matches = list(Path("/kaggle/input").rglob(filename))

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {filename}, found {len(matches)}:\n"
            + "\n".join(map(str, matches))
        )

    return matches[0]


SANA_PATH = find_one("sana_class_embeddings.pt")

sana_bundle = torch.load(
    SANA_PATH,
    map_location="cpu",
    weights_only=False,
)

required_keys = {
    "class_names",
    "templates",
    "embeddings",
    "class_spec",
    "model_id",
    "representation",
}

missing = required_keys - set(sana_bundle)
assert not missing, f"Missing SANA keys: {missing}"

class_names = list(sana_bundle["class_names"])
templates = list(sana_bundle["templates"])
sana_embeddings = sana_bundle["embeddings"].float()
class_spec = sana_bundle["class_spec"]

assert sana_embeddings.ndim == 2
assert sana_embeddings.shape[0] == len(class_names)
assert len(class_names) == class_spec["class_count"]
assert len(templates) == class_spec["template_count"]
assert class_names == class_spec["class_names"]
assert templates == class_spec["templates"]
assert len(set(class_names)) == len(class_names)
assert all(template.count("{}") == 1 for template in templates)
assert torch.isfinite(sana_embeddings).all()

class_spec_content = json.dumps(
    class_spec,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

class_spec_sha256 = hashlib.sha256(
    class_spec_content
).hexdigest()

print("SANA file:", SANA_PATH)
print("Classes:", len(class_names))
print("Templates per class:", len(templates))
print("SANA embeddings:", tuple(sana_embeddings.shape))
print("SANA model:", sana_bundle["model_id"])
print("Class specification SHA-256:", class_spec_sha256)
print("\nFirst classes:", class_names[:10])
print("\nTemplates:")
for template in templates:
    print(" ", template)

SANA file: /kaggle/input/datasets/aveenhussein/classes/sana_class_embeddings.pt
Classes: 3000
Templates per class: 5
SANA embeddings: (3000, 2304)
SANA model: Efficient-Large-Model/gemma-2-2b-it
Class specification SHA-256: 9c8341e536823b3a70ec56f8a34217afca315447dc22e185bf83a05701bc5999

First classes: ['entity', 'physical entity', 'abstraction', 'thing', 'object', 'whole', 'congener', 'living thing', 'organism', 'benthos']

Templates:
  a bad photo of a {}.
  a photo of many {}.
  a sculpture of a {}.
  a photo of the hard to see {}.
  a low resolution photo of the {}.


In [3]:
from transformers import (
    AutoProcessor,
    Qwen3VLForConditionalGeneration,
)

VLM_MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = Qwen3VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype="auto",
    device_map="auto" if torch.cuda.is_available() else None,
)

processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)
tokenizer = processor.tokenizer
model.eval()

assert tokenizer.is_fast, (
    "A fast tokenizer is required for exact class-token spans."
)


def get_text_embedding_layer(model):
    candidates = [
        lambda m: m.model.language_model.embed_tokens,
        lambda m: m.language_model.model.embed_tokens,
        lambda m: m.language_model.embed_tokens,
        lambda m: m.get_input_embeddings(),
    ]

    for getter in candidates:
        try:
            layer = getter(model)
            if layer is not None:
                return layer
        except AttributeError:
            pass

    raise AttributeError(
        "Could not locate the VLM text embedding layer."
    )


embed_tokens = get_text_embedding_layer(model)
embed_device = next(embed_tokens.parameters()).device

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = (
        tokenizer.eos_token or tokenizer.unk_token
    )

tokenizer.padding_side = "right"

print("VLM:", VLM_MODEL_ID)
print("Embedding device:", embed_device)
print("VLM hidden dimension:", embed_tokens.weight.shape[1])

ImportError: huggingface-hub>=0.34.0,<1.0 is required for a normal functioning of this module, but found huggingface-hub==1.11.0.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

# code

In [10]:
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from IPython.display import FileLink, display

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

matches = list(
    Path("/kaggle/working").rglob(
        "llava15_sana_class_projection_inputs.pt"
    )
)

assert len(matches) == 1, (
    "Expected exactly one paired projection file, "
    f"found: {matches}"
)

BUNDLE_PATH = matches[0]

bundle = torch.load(
    BUNDLE_PATH,
    map_location="cpu",
    weights_only=False,
)

required_keys = {
    "X",
    "Y",
    "class_names",
    "templates",
    "ids",
    "class_spec",
    "class_spec_sha256",
    "source_model",
    "target_model",
}

missing = required_keys - set(bundle)
assert not missing, f"Missing keys: {missing}"

X_raw = bundle["X"].float().to(device)
Y_raw = bundle["Y"].float().to(device)

class_names = list(bundle["class_names"])
templates = list(bundle["templates"])
class_ids = list(bundle["ids"])

assert X_raw.ndim == 2
assert Y_raw.ndim == 2
assert X_raw.shape[0] == Y_raw.shape[0]
assert X_raw.shape[0] == len(class_names)
assert class_names == bundle["class_spec"]["class_names"]
assert templates == bundle["class_spec"]["templates"]
assert torch.isfinite(X_raw).all()
assert torch.isfinite(Y_raw).all()

print("Device:", device)
print("Bundle:", BUNDLE_PATH)
print("X / LLaVA:", tuple(X_raw.shape))
print("Y / SANA:", tuple(Y_raw.shape))
print("Classes:", len(class_names))
print("Templates:", len(templates))
print("Source:", bundle["source_model"])
print("Target:", bundle["target_model"])

Device: cpu
Bundle: /kaggle/working/llava15_sana_class_projection_inputs.pt
X / LLaVA: (3000, 4096)
Y / SANA: (3000, 2304)
Classes: 3000
Templates: 5
Source: llava-hf/llava-1.5-7b-hf
Target: Efficient-Large-Model/gemma-2-2b-it


In [11]:
def normalize_rows(tensor):
    return F.normalize(
        tensor,
        dim=1,
        eps=1e-12,
    )


# Source preprocessing.
X_unit = normalize_rows(X_raw)

X_center_mean = X_unit.mean(
    dim=0,
    keepdim=True,
)

X_model = normalize_rows(
    X_unit - X_center_mean
)


# Separate SANA direction and magnitude.
Y_raw_norms = Y_raw.norm(
    dim=1,
    keepdim=True,
).clamp_min(1e-12)

Y_direction = Y_raw / Y_raw_norms

Y_direction_mean = Y_direction.mean(
    dim=0,
    keepdim=True,
)

Y_direction_centered = (
    Y_direction - Y_direction_mean
)

Y_covariance = (
    Y_direction_centered.T
    @ Y_direction_centered
) / max(Y_direction.shape[0] - 1, 1)

Y_covariance = (
    Y_covariance + Y_covariance.T
) * 0.5

identity_y = torch.eye(
    Y_covariance.shape[0],
    device=device,
    dtype=Y_covariance.dtype,
)

Y_cholesky = None
Y_cholesky_epsilon = None

for epsilon in [
    1e-5,
    3e-5,
    1e-4,
    3e-4,
    1e-3,
]:
    candidate, info = torch.linalg.cholesky_ex(
        Y_covariance + epsilon * identity_y
    )

    if int(info.max()) == 0:
        Y_cholesky = candidate
        Y_cholesky_epsilon = epsilon
        break

assert Y_cholesky is not None, (
    "Could not construct the SANA whitener."
)

Y_white = torch.linalg.solve_triangular(
    Y_cholesky,
    Y_direction_centered.T,
    upper=False,
).T


# Learn SANA magnitude separately.
Y_log_norm = Y_raw_norms.log()

Y_log_norm_mean = Y_log_norm.mean(
    dim=0,
    keepdim=True,
)

Y_log_norm_std = Y_log_norm.std(
    dim=0,
    keepdim=True,
).clamp_min(1e-6)

Y_log_norm_z = (
    Y_log_norm - Y_log_norm_mean
) / Y_log_norm_std

Y_norm_min = float(Y_raw_norms.min())
Y_norm_max = float(Y_raw_norms.max())

assert torch.isfinite(X_model).all()
assert torch.isfinite(Y_white).all()
assert torch.isfinite(Y_log_norm_z).all()

print("X model:", tuple(X_model.shape))
print("Y whitened:", tuple(Y_white.shape))
print("Y norm target:", tuple(Y_log_norm_z.shape))
print("Whitening epsilon:", Y_cholesky_epsilon)
print("Y norm range:", Y_norm_min, Y_norm_max)

X model: (3000, 4096)
Y whitened: (3000, 2304)
Y norm target: (3000, 1)
Whitening epsilon: 1e-05
Y norm range: 85.95154571533203 238.76649475097656


In [12]:
def polynomial_kernel(
    first,
    second,
    sigma,
    degree,
    coef0,
):
    return (
        sigma * (first @ second.T) + coef0
    ) ** degree


sigma = 0.5003
degree = 2
coef0 = 0.2
requested_lambda = 1e-3

print("Building calculation kernel...")

fit_start = time.perf_counter()

K = polynomial_kernel(
    X_model,
    X_model,
    sigma=sigma,
    degree=degree,
    coef0=coef0,
)

K = (K + K.T) * 0.5

identity_k = torch.eye(
    K.shape[0],
    device=device,
    dtype=K.dtype,
)

# Solve direction and norm using one decomposition.
rhs = torch.cat(
    [Y_white, Y_log_norm_z],
    dim=1,
)

kernel_cholesky = None
lambda_reg = None

for multiplier in [1, 3, 10, 30, 100, 300]:
    candidate_lambda = (
        requested_lambda * multiplier
    )

    candidate, info = torch.linalg.cholesky_ex(
        K + candidate_lambda * identity_k
    )

    if int(info.max()) == 0:
        kernel_cholesky = candidate
        lambda_reg = candidate_lambda
        break

assert kernel_cholesky is not None, (
    "Could not solve the regularized kernel system."
)

coefficients = torch.cholesky_solve(
    rhs,
    kernel_cholesky,
)

target_dimension = Y_white.shape[1]

direction_coefficients = coefficients[
    :, :target_dimension
]

norm_coefficients = coefficients[
    :, target_dimension:
]

fit_seconds = time.perf_counter() - fit_start

print("Kernel:", tuple(K.shape))
print(
    "Direction coefficients:",
    tuple(direction_coefficients.shape),
)
print(
    "Norm coefficients:",
    tuple(norm_coefficients.shape),
)
print("Effective lambda:", lambda_reg)
print("Fit seconds:", round(fit_seconds, 2))

Building calculation kernel...
Kernel: (3000, 3000)
Direction coefficients: (3000, 2304)
Norm coefficients: (3000, 1)
Effective lambda: 0.001
Fit seconds: 1.23


In [13]:
with torch.inference_mode():
    predicted_white = (
        K @ direction_coefficients
    )

    predicted_direction = (
        predicted_white @ Y_cholesky.T
        + Y_direction_mean
    )

    predicted_direction = normalize_rows(
        predicted_direction
    )

    predicted_log_norm_z = (
        K @ norm_coefficients
    )

    predicted_log_norm = (
        predicted_log_norm_z * Y_log_norm_std
        + Y_log_norm_mean
    )

    predicted_norm = (
        predicted_log_norm.exp()
        .clamp(
            min=Y_norm_min,
            max=Y_norm_max,
        )
    )

    predicted_raw = (
        predicted_direction * predicted_norm
    )


target_direction = normalize_rows(Y_raw)
prediction_direction = normalize_rows(predicted_raw)

correct_cosines = F.cosine_similarity(
    prediction_direction,
    target_direction,
    dim=1,
)

normalized_mse = F.mse_loss(
    prediction_direction,
    target_direction,
)

raw_normalized_mse = (
    F.mse_loss(predicted_raw, Y_raw)
    / Y_raw.pow(2).mean().clamp_min(1e-12)
)


# Retrieval metrics in chunks.
retrieval_batch_size = 256
wrong_similarity_sum = 0.0
wrong_similarity_count = 0
top1_correct = 0

for start in range(
    0,
    len(class_names),
    retrieval_batch_size,
):
    end = min(
        start + retrieval_batch_size,
        len(class_names),
    )

    similarities = (
        prediction_direction[start:end]
        @ target_direction.T
    )

    local_rows = torch.arange(
        end - start,
        device=device,
    )

    correct_columns = torch.arange(
        start,
        end,
        device=device,
    )

    diagonal_values = similarities[
        local_rows,
        correct_columns,
    ]

    wrong_similarity_sum += float(
        similarities.sum() - diagonal_values.sum()
    )

    wrong_similarity_count += (
        similarities.numel() - len(diagonal_values)
    )

    top1_correct += int(
        (
            similarities.argmax(dim=1)
            == correct_columns
        ).sum()
    )


correct_cosine = float(correct_cosines.mean())
wrong_cosine = (
    wrong_similarity_sum / wrong_similarity_count
)
separation_gap = correct_cosine - wrong_cosine
top1 = top1_correct / len(class_names)

metrics = {
    "correct_cosine": correct_cosine,
    "wrong_cosine": wrong_cosine,
    "separation_gap": separation_gap,
    "top1": top1,
    "normalized_mse": float(normalized_mse),
    "raw_normalized_mse": float(
        raw_normalized_mse
    ),
}

print("Calculation-set projection metrics")
print("==================================")
print("Correct cosine:", metrics["correct_cosine"])
print("Wrong cosine:", metrics["wrong_cosine"])
print("Separation gap:", metrics["separation_gap"])
print("Top-1:", metrics["top1"])
print("Normalized MSE:", metrics["normalized_mse"])
print(
    "Raw normalized MSE:",
    metrics["raw_normalized_mse"],
)

Calculation-set projection metrics
Correct cosine: 0.9999979138374329
Wrong cosine: 0.6082077046515505
Separation gap: 0.39179020918588237
Top-1: 1.0
Normalized MSE: 1.8615914365582853e-09
Raw normalized MSE: 4.811439794139005e-06


In [14]:
OUTPUT_PATH = Path(
    "/kaggle/working/"
    "sana_llava_class_kernel_projector.pt"
)

projector_bundle = {
    "format_version": 1,
    "kind": "sana_class_kernel_projector",

    "source_model": bundle["source_model"],
    "target_model": bundle["target_model"],
    "source_representation": bundle[
        "source_representation"
    ],
    "target_representation": bundle[
        "target_representation"
    ],

    "class_names": class_names,
    "templates": templates,
    "ids": class_ids,
    "class_spec": bundle["class_spec"],
    "class_spec_sha256": bundle[
        "class_spec_sha256"
    ],

    # Source preprocessing.
    "X_center_mean": X_center_mean.cpu(),
    "train_features": X_model.cpu(),

    # Kernel.
    "kernel": {
        "kind": "polynomial",
        "sigma": sigma,
        "degree": degree,
        "coef0": coef0,
        "lambda_reg": lambda_reg,
    },

    # Learned mapping.
    "direction_coefficients": (
        direction_coefficients.cpu()
    ),
    "norm_coefficients": (
        norm_coefficients.cpu()
    ),

    # SANA direction reconstruction.
    "Y_direction_mean": (
        Y_direction_mean.cpu()
    ),
    "Y_cholesky": Y_cholesky.cpu(),
    "Y_cholesky_epsilon": (
        Y_cholesky_epsilon
    ),

    # SANA magnitude reconstruction.
    "Y_log_norm_mean": (
        Y_log_norm_mean.cpu()
    ),
    "Y_log_norm_std": (
        Y_log_norm_std.cpu()
    ),
    "Y_norm_min": Y_norm_min,
    "Y_norm_max": Y_norm_max,

    "training_metrics": metrics,
    "fit_seconds": fit_seconds,
}

torch.save(
    projector_bundle,
    OUTPUT_PATH,
)

saved = torch.load(
    OUTPUT_PATH,
    map_location="cpu",
    weights_only=False,
)

assert saved["train_features"].shape == X_model.shape
assert (
    saved["direction_coefficients"].shape
    == direction_coefficients.shape
)
assert saved["class_names"] == class_names
assert saved["class_spec_sha256"] == bundle[
    "class_spec_sha256"
]

print("Saved and verified:", OUTPUT_PATH)
print(
    "Training features:",
    tuple(saved["train_features"].shape),
)
print(
    "Direction coefficients:",
    tuple(
        saved["direction_coefficients"].shape
    ),
)
print("Metrics:", saved["training_metrics"])

display(FileLink(str(OUTPUT_PATH)))

Saved and verified: /kaggle/working/sana_llava_class_kernel_projector.pt
Training features: (3000, 4096)
Direction coefficients: (3000, 2304)
Metrics: {'correct_cosine': 0.9999979138374329, 'wrong_cosine': 0.6082077046515505, 'separation_gap': 0.39179020918588237, 'top1': 1.0, 'normalized_mse': 1.8615914365582853e-09, 'raw_normalized_mse': 4.811439794139005e-06}


/kaggle/working/sana_llava_class_kernel_projector.pt